# ModelNet40 PointNeXt 训练 Notebook

这个 Notebook 使用项目中的 `src/pointnext_demo` 模块完成数据读取、模型构建、训练、验证和预测。默认启用 GPU 加速检测；如果 PyTorch 不能访问 NVIDIA GPU，会自动回落到 CPU。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class TrainConfig:
    data_root: str = "modelnet40_train_data/modelnet40_normal_resampled"
    labels: str = "labels/modelnet40.txt"
    out_dir: str = "runs/notebook_pointnext_s_c64_normals"
    variant: str = "s"
    width: int = 64
    nsample: int = 32
    num_points: int = 1024
    use_normals: bool = True
    random_rotate: bool = False
    epochs: int = 600
    batch_size: int = 32
    lr: float = 1e-3
    weight_decay: float = 5e-2
    label_smoothing: float = 0.2
    val_ratio: float = 0.15
    num_workers: int = 0
    seed: int = 42
    use_gpu: bool = True

cfg = TrainConfig()
cfg

如果只是验证流程，可以临时把 `epochs`、`width`、`num_points` 调小，例如：

```python
cfg.epochs = 1
cfg.width = 8
cfg.nsample = 8
cfg.num_points = 64
cfg.batch_size = 4
cfg.data_root = "data_toy"
```

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

from src.pointnext_demo.data import ModelNetLikeDataset, collate_batch
from src.pointnext_demo.model import build_model
from src.pointnext_demo.train import run_epoch, stratified_split
from src.pointnext_demo.utils import load_labels, save_checkpoint, save_json, set_seed

set_seed(cfg.seed)
device = torch.device("cuda" if cfg.use_gpu and torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))
device

In [ ]:
labels = load_labels(PROJECT_ROOT / cfg.labels)

train_dataset = ModelNetLikeDataset(
    PROJECT_ROOT / cfg.data_root,
    split="train",
    labels=labels,
    num_points=cfg.num_points,
    use_normals=cfg.use_normals,
    train=True,
    random_rotate=cfg.random_rotate,
)
val_dataset = ModelNetLikeDataset(
    PROJECT_ROOT / cfg.data_root,
    split="train",
    labels=labels,
    num_points=cfg.num_points,
    use_normals=cfg.use_normals,
    train=False,
)

train_indices, val_indices = stratified_split(train_dataset, cfg.val_ratio, cfg.seed)
train_set = Subset(train_dataset, train_indices)
val_set = Subset(val_dataset, val_indices)

train_loader = DataLoader(
    train_set,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    collate_fn=collate_batch,
    drop_last=len(train_set) > cfg.batch_size,
)
val_loader = DataLoader(
    val_set,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collate_batch,
)

{
    "samples": len(train_dataset),
    "train": len(train_set),
    "val": len(val_set),
    "classes": len(labels),
    "channels": train_dataset.num_channels,
    "device": str(device),
}

In [ ]:
model = build_model(
    cfg.variant,
    num_classes=len(labels),
    use_normals=cfg.use_normals,
    width=cfg.width,
    nsample=cfg.nsample,
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

sum(p.numel() for p in model.parameters())

In [ ]:
out_dir = PROJECT_ROOT / cfg.out_dir
out_dir.mkdir(parents=True, exist_ok=True)

best_acc = 0.0
best_class_acc = 0.0
history = []

for epoch in range(1, cfg.epochs + 1):
    train_loss, train_acc, train_class_acc = run_epoch(
        model, train_loader, criterion, optimizer, device, train=True, num_classes=len(labels)
    )
    val_loss, val_acc, val_class_acc = run_epoch(
        model, val_loader, criterion, optimizer, device, train=False, num_classes=len(labels)
    )
    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_class_acc": train_class_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_class_acc": val_class_acc,
        "lr": scheduler.get_last_lr()[0],
    }
    history.append(row)
    print(
        f"epoch {epoch:03d}: train_loss={train_loss:.4f} "
        f"train_instance_acc={train_acc:.4f} train_class_acc={train_class_acc:.4f} "
        f"val_loss={val_loss:.4f} val_instance_acc={val_acc:.4f} val_class_acc={val_class_acc:.4f}"
    )

    if val_acc >= best_acc:
        best_acc = val_acc
        best_class_acc = val_class_acc
        save_checkpoint(
            out_dir / "best.pt",
            {
                "model": model.state_dict(),
                "labels": labels,
                "args": asdict(cfg),
                "num_classes": len(labels),
                "num_channels": train_dataset.num_channels,
                "train_indices": train_indices,
                "val_indices": val_indices,
                "best_acc": best_acc,
                "best_class_acc": best_class_acc,
                "epoch": epoch,
            },
        )

save_json(out_dir / "history.json", {"history": history, "best_acc": best_acc, "best_class_acc": best_class_acc})
{"best_instance_acc": best_acc, "best_class_acc": best_class_acc}

## 预测导出

如果存在 `test` split，可以在下面单元格中导出预测 CSV。若当前只有训练目录，可先保持该单元格不运行。

In [ ]:
# 也可以继续使用命令行预测：
# python -m src.pointnext_demo.predict --checkpoint runs/notebook_pointnext_s_c64_normals/best.pt --out-csv submit.csv --votes 10

checkpoint_path = out_dir / "best.pt"
checkpoint_path